# Машинне навчання — лабораторна робота 2

## Завдання

Натренувати DecisionTreeClassifier та RandomForestClassifier для даного набору даних. Знайти оптимальні значення для гіперпараметрів. Порівняти результати.

Імпортуємо бібліотеки:

In [1]:
import os
import kagglehub
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

Завантажуємо дані:

In [2]:
path = kagglehub.dataset_download("yaaryiitturan/pendigits")
df = pd.read_csv(os.path.join(path, "pendigits_txt.csv"), header=None)

df.columns = [f"input{i}" for i in range(1, 17)] + ["class"]

df = df.iloc[1:].reset_index(drop=True)
df = df.astype(int)

X = df.drop(columns=["class"])
y = df["class"]

df.head()

,input1,input2,input3,input4,input5,input6,input7,input8,input9,input10,input11,input12,input13,input14,input15,input16,class
0,47,100,27,81,57,37,26,0,0,23,56,53,100,90,40,98,8
1,0,89,27,100,42,75,29,45,15,15,37,0,69,2,100,6,2
2,0,57,31,68,72,90,100,100,76,75,50,51,28,25,16,0,1
3,0,100,7,92,5,68,19,45,86,34,100,45,74,23,67,0,4
4,0,67,49,83,100,100,81,80,60,60,40,40,33,20,47,0,1


Ділимо дані на навчальну та тестову вибірки:

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)
print("Розподіл класів у train:")
print(y_train.value_counts(normalize=True).sort_index().round(3))

Train: (8793, 16) (8793,)
Test: (2199, 16) (2199,)
Розподіл класів у train:
class
0    0.104
1    0.104
2    0.104
3    0.096
4    0.104
5    0.096
6    0.096
7    0.104
8    0.096
9    0.096
Name: proportion, dtype: float64


Підбираємо гіперпараметри для Decision Tree:

In [4]:
dt = DecisionTreeClassifier(random_state=42)

dt_grid = {
    "criterion": ["gini", "entropy", "log_loss"],
    "max_depth": [None, 10, 20, 30, 40],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

dt_search = GridSearchCV(
    dt,
    dt_grid,
    cv=cv,
    n_jobs=-1,
    scoring="accuracy"
)

dt_search.fit(X_train, y_train)

dt_best = dt_search.best_estimator_
dt_pred = dt_best.predict(X_test)

print("Найкращі параметри:", dt_search.best_params_)
print("CV accuracy:", round(dt_search.best_score_, 4))
print("Test accuracy:", round(accuracy_score(y_test, dt_pred), 4))
print(classification_report(y_test, dt_pred))

Найкращі параметри: {'criterion': 'entropy', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}
CV accuracy: 0.9577
Test accuracy: 0.9582
              precision    recall  f1-score   support

           0       0.98      0.99      0.98       229
           1       0.94      0.90      0.92       229
           2       0.95      0.97      0.96       229
           3       0.96      0.95      0.95       211
           4       0.98      0.96      0.97       229
           5       0.94      0.95      0.95       211
           6       0.99      1.00      0.99       211
           7       0.93      0.98      0.96       228
           8       0.98      0.97      0.97       211
           9       0.93      0.91      0.92       211

    accuracy                           0.96      2199
   macro avg       0.96      0.96      0.96      2199
weighted avg       0.96      0.96      0.96      2199



Підбираємо гіперпараметри для Random Forest:

In [5]:
rf = RandomForestClassifier(random_state=42)

rf_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 20, 30, 40],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

rf_search = GridSearchCV(
    rf,
    rf_grid,
    cv=cv,
    n_jobs=-1,
    scoring="accuracy"
)

rf_search.fit(X_train, y_train)

rf_best = rf_search.best_estimator_
rf_pred = rf_best.predict(X_test)

print("Найкращі параметри:", rf_search.best_params_)
print("CV accuracy:", round(rf_search.best_score_, 4))
print("Test accuracy:", round(accuracy_score(y_test, rf_pred), 4))
print(classification_report(y_test, rf_pred))

Найкращі параметри: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
CV accuracy: 0.99
Test accuracy: 0.9914
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       229
           1       0.99      0.97      0.98       229
           2       0.97      0.99      0.98       229
           3       0.99      0.99      0.99       211
           4       0.99      1.00      0.99       229
           5       1.00      1.00      1.00       211
           6       1.00      1.00      1.00       211
           7       0.98      1.00      0.99       228
           8       1.00      0.99      0.99       211
           9       1.00      0.99      0.99       211

    accuracy                           0.99      2199
   macro avg       0.99      0.99      0.99      2199
weighted avg       0.99      0.99      0.99      2199



Порівнюємо результати обох моделей на тестовій вибірці:

In [6]:
results = pd.DataFrame([
    {
        "model": "Decision Tree",
        "cv_accuracy": dt_search.best_score_,
        "test_accuracy": accuracy_score(y_test, dt_pred),
    },
    {
        "model": "Random Forest",
        "cv_accuracy": rf_search.best_score_,
        "test_accuracy": accuracy_score(y_test, rf_pred),
    }
])

results.sort_values("test_accuracy", ascending=False).reset_index(drop=True)

,model,cv_accuracy,test_accuracy
0,Random Forest,0.989992,0.991360
1,Decision Tree,0.957693,0.958163


У цьому датасеті кращий результат показує Random Forest.